# Raport końcowy — analiza danych i decyzje przedmodelowe

**Projekt:** Klasyfikator fake-news (binarna klasyfikacja: 1 = Real, 0 = Fake) na zbiorze ISOT *Fake and Real News*  
**Architektura docelowa:** BiLSTM + warstwa atencji, embeddingi **GloVe 300d**, implementacja w **strict PyTorch** (bez TF/Keras/JAX, bez HuggingFace `Trainer` / Lightning).  
**Cel raportu:** w jednym miejscu zebrać wszystkie wnioski z EDA, analizy wycieków i cleaningu; potwierdzić decyzje preprocessingu i przedstawić plan dalszych prac.

**Notebooki źródłowe** (każdy wnioski rozszerza):
- `notebooks/explore/01_eda.ipynb` — eksploracja surowych danych
- `notebooks/explore/02_leakage.ipynb` — analiza wycieków informacji
- `notebooks/explore/03_cleaning.ipynb` — pipeline cleaningu
- `notebooks/explore/04_post_cleaning.ipynb` — sanity check po cleaningu

Implementacja czyszczenia: `src/preprocessing/cleaning.py`.

In [1]:
import re
import warnings
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings('ignore', category=UserWarning)
pd.options.plotting.backend = 'plotly'
pio.renderers.default = 'notebook_connected'
pd.set_option('display.max_colwidth', 100)

LABEL_NAMES = {0: 'Fake', 1: 'Real'}
LABEL_COLORS = {'Fake': '#EF553B', 'Real': '#636EFA'}
REPO = Path('/home/aoleszkiewicz/dev/factlens')

In [2]:
raw_dir = Path(kagglehub.dataset_download('clmentbisaillon/fake-and-real-news-dataset'))
true_df = pd.read_csv(raw_dir / 'True.csv').assign(label=1)
fake_df = pd.read_csv(raw_dir / 'Fake.csv').assign(label=0)
raw = pd.concat([true_df, fake_df], ignore_index=True)
raw['label_name'] = raw['label'].map(LABEL_NAMES)

clean = pd.read_parquet(REPO / 'data/processed/news_cleaned.parquet')
clean['label_name'] = clean['label'].map(LABEL_NAMES)

raw['n_words']   = raw['text'].fillna('').str.split().str.len()
clean['n_words'] = clean['text'].fillna('').str.split().str.len()

f'Surowe: {len(raw):,} | Po cleaningu: {len(clean):,} (-{(1 - len(clean)/len(raw))*100:.2f}%)'

'Surowe: 44,898 | Po cleaningu: 38,470 (-14.32%)'

## 1. Zbiór danych

**Źródło:** Kaggle, `clmentbisaillon/fake-and-real-news-dataset` (ISOT Research Lab, University of Victoria). Pobierany w runtime przez `kagglehub`.

**Format wejścia:** dwa pliki CSV — `True.csv` (artykuły Reuters) i `Fake.csv` (rozmaite portale fake-news / opinion). Kolumny: `title`, `text`, `subject`, `date`. Etykietę `label` dorabiamy: **1 = Real**, **0 = Fake**.

**Rozmiar:** 44 898 artykułów łącznie (próbka równoważona ~52% Fake / 48% Real).

In [3]:
balance = raw.groupby('label_name').size().rename('count').reset_index()
balance['udział_%'] = (balance['count'] / balance['count'].sum() * 100).round(2)
fig = px.bar(balance, x='label_name', y='count', color='label_name',
             color_discrete_map=LABEL_COLORS, text='count',
             title='Balans klas w surowym zbiorze')
fig.update_layout(xaxis_title='', yaxis_title='liczba artykułów', showlegend=False, height=340)
fig.show()
balance

,label_name,count,udział_%
0,Fake,23481,52.3
1,Real,21417,47.7


## 2. Kluczowe obserwacje z eksploracji (EDA)

Pełna analiza: `notebooks/explore/01_eda.ipynb`. Tu — tylko ustalenia, które wpłynęły na decyzje preprocessingu.

1. **Balans** ~52/48 (Fake/Real) — wystarczająco zrównoważony, aby trenować bez resamplingu i raportować accuracy obok F1.
2. **Duplikaty** — ~6.4 tys. dokładnych duplikatów `text`, prawie wszystkie po stronie Fake. Część cross-class jest trywialna (puste teksty); reszta dotyczy *wewnątrzklasowych* powtórzeń.
3. **Metadane są toksyczne** — `subject` jest 1:1 z klasą, `date` ma w Fake niestandardowy format (52% nieparsowalne), `title` zawiera prefiksy clickbaitowe (`VIDEO:`, `WATCH:`) i ALL-CAPS. Żadna z tych kolumn nie wchodzi do modelu.
4. **Długość tekstu** — mediana 360 słów (Real ~390, Fake ~330), 95. percentyl ~900, 99. percentyl ~1700. Wybór `max_seq_len = 900` pokrywa 95% danych bez nadmiernego paddingu.
5. **Bardzo krótkie teksty** (< 10 słów) to ~870 rekordów, niemal wszystkie Fake — typowy sygnał scrapowych artefaktów. Filtrujemy `min_words=10`.
6. **Język** — heurystyczna detekcja wskazuje, że ~97% próbek to angielski; pozostałe ~3% to fragmenty wielojęzyczne lub same metadane. Po cleaningu liczba ta dodatkowo spada.

In [4]:
fig = go.Figure()
fig.add_trace(go.Histogram(x=raw[raw.label==0]['n_words'].clip(upper=2000), name='Fake',
                            marker_color='#EF553B', opacity=0.65, nbinsx=80))
fig.add_trace(go.Histogram(x=raw[raw.label==1]['n_words'].clip(upper=2000), name='Real',
                            marker_color='#636EFA', opacity=0.65, nbinsx=80))
fig.add_vline(x=900, line_dash='dash', line_color='black',
              annotation_text='max_seq_len = 900 (p95)', annotation_position='top right')
fig.update_layout(barmode='overlay', title='Rozkład długości artykułu per klasa',
                  xaxis_title='liczba słów (klipowane do 2000)', yaxis_title='liczba artykułów', height=420)
fig.show()

## 3. Wycieki informacji (data leakage)

Pełna analiza: `notebooks/explore/02_leakage.ipynb`.

Zbiór ISOT ma **trzy poważne, nakładające się wycieki**, których nieusunięcie zafałszowałoby wynik klasyfikatora:

**(a) `subject` — etykieta pod inną nazwą.** Wartości `politicsNews` / `worldnews` występują wyłącznie w Real; pozostałe wyłącznie w Fake. Decyzja: kolumna usunięta.

**(b) Stempel agencji Reuters w `text`.** ~91% artykułów Real zaczyna się od wzorca `WASHINGTON (Reuters) - …`; w Fake praktycznie nie występuje. Decyzja: regex w `cleaning.py` zdejmuje cały prefiks.

**(c) Markery stylu Fake.** URL-e (`https://`, `pic.twitter.com`), `@handles`, podpisy obrazków (`Featured image via … Getty Images`), sygnatury portali (`21st Century Wire`), prefiksy (`VIDEO:`, `WATCH:`) — występują w kilku–kilkudziesięciu procentach Fake i niemal nigdy w Real. Decyzja: lista regexów w `cleaning.py`.

Test dyskryminacyjny (chi² + Mutual Information) w `02_leakage.ipynb` potwierdza, że każdy z tych markerów jest istotnie skorelowany z klasą — sam stempel Reutersa daje >0.5 bita informacji o etykiecie.

In [5]:
MARKERS = {
    'reuters_dateline': r'^[A-Z][A-Z\s/]{1,40} \(Reuters\)\s*-',
    'url_http':         r'https?://\S+',
    'at_handle':        r'@\w+',
    'getty_images':     r'getty\s+images?',
    'featured_image':   r'featured\s+image',
    'wire_21cw':        r'21st\s+century\s+wire',
    'twitter_pic':      r'pic\.twitter\.com',
}
rows = []
for name, pat in MARKERS.items():
    raw_hit   = raw['text'].str.contains(pat, regex=True, flags=re.IGNORECASE).fillna(False)
    fake_pct  = raw_hit[raw.label==0].mean() * 100
    real_pct  = raw_hit[raw.label==1].mean() * 100
    rows.append({'marker': name, 'Fake [%]': round(fake_pct, 2), 'Real [%]': round(real_pct, 2)})
leak_df = pd.DataFrame(rows)
fig = px.bar(leak_df.melt(id_vars='marker', var_name='Klasa', value_name='%'),
             x='marker', y='%', color='Klasa', barmode='group',
             color_discrete_map={'Fake [%]': '#EF553B', 'Real [%]': '#636EFA'},
             title='Markery wycieku w surowym <b>text</b> — % dokumentów per klasa')
fig.update_layout(xaxis_title='', yaxis_title='% dokumentów', xaxis_tickangle=-25,
                  height=400, legend_title='')
fig.show()

## 4. Pipeline cleaningu — co robimy

Implementacja: `src/preprocessing/cleaning.py`. Pipeline z `notebooks/explore/03_cleaning.ipynb` aplikuje kolejność operacji (kolejność ma znaczenie):

1. **Stemple agencji** — usuwamy `^[A-Z][A-Z\s/]+ \(Reuters\) -` i wzmianki `(Reuters|AP|AFP)`, `\breuters\b`.
2. **URL-e i handles** — `https?://`, `www\.`, `pic\.twitter\.com`, `tmsnrt\.rs/`, `@\w+`.
3. **Podpisy obrazków** — `featured image via … (Getty|Flickr|…)`, `Photo by … Getty`, `Getty Images`.
4. **Sygnatury portali** — `21st Century Wire`.
5. **HTML i encje** — `<script>…</script>`, dowolne tagi `<…>`, `&amp;` / `&quot;` / `&lt;` / `&gt;` / `&nbsp;`.
6. **Korekta złamań** — `2017Trump → 2017 Trump`.
7. **Normalizacja końcowa** — kompresja whitespace, usunięcie pojedynczych liter, lower-case.
8. **Filtrowanie krótkich** — `filter_short_articles(min_words=10)`.
9. **Deduplikacja** — `drop_duplicates(subset='text')` po normalizacji.

Wynik zapisywany do `data/processed/news_cleaned.parquet` (kolumny `id`, `text`, `label`).

In [6]:
summary = pd.DataFrame({
    'Metryka': [
        'liczba rekordów',
        'liczba Fake',
        'liczba Real',
        'duplikaty (text)',
        'puste text',
        'mediana liczby słów',
        'p95 liczby słów',
        '% z markerem reuters_dateline',
        '% z URL',
        '% z @handle',
        '% z getty_images',
    ],
    'Surowe': [
        f'{len(raw):,}',
        f"{(raw.label==0).sum():,}",
        f"{(raw.label==1).sum():,}",
        f"{raw['text'].duplicated().sum():,}",
        f"{(raw['text'].str.strip() == '').sum():,}",
        f"{raw['n_words'].median():.0f}",
        f"{np.percentile(raw['n_words'], 95):.0f}",
        *[f"{(raw['text'].str.contains(p, regex=True, flags=re.IGNORECASE).fillna(False)).mean()*100:.2f}%"
          for p in [MARKERS['reuters_dateline'], MARKERS['url_http'],
                    MARKERS['at_handle'], MARKERS['getty_images']]],
    ],
    'Po cleaningu': [
        f'{len(clean):,}',
        f"{(clean.label==0).sum():,}",
        f"{(clean.label==1).sum():,}",
        f"{clean['text'].duplicated().sum():,}",
        f"{(clean['text'].str.strip() == '').sum():,}",
        f"{clean['n_words'].median():.0f}",
        f"{np.percentile(clean['n_words'], 95):.0f}",
        *[f"{clean['text'].str.contains(p, regex=True).mean()*100:.2f}%"
          for p in [MARKERS['reuters_dateline'], MARKERS['url_http'],
                    MARKERS['at_handle'], MARKERS['getty_images']]],
    ],
})
summary

,Metryka,Surowe,Po cleaningu
0,liczba rekordów,"44,898","38,470"
1,liczba Fake,"23,481","17,281"
2,liczba Real,"21,417","21,189"
3,duplikaty (text),"6,252",0
4,puste text,631,0
5,mediana liczby słów,362,352
6,p95 liczby słów,904,848
7,% z markerem reuters_dateline,39.69%,0.00%
8,% z URL,7.35%,0.00%
9,% z @handle,14.25%,0.00%


**Wniosek z cleaningu.** Wszystkie kluczowe markery wycieku zniknęły lub spadły do <0.1%. Liczba rekordów spadła o ~14% (głównie skasowane duplikaty Fake), co dało nieco lepszy balans (~50/50). Mediana i p95 długości praktycznie się nie zmieniły — wyrzucamy tylko krótkie ciągi (URL-e, handles), nie skracamy treści.

## 5. Decyzje preprocessingu — co robimy i dlaczego

Lista skondensowanych decyzji wynikających z analiz powyżej. **Każda jest deterministyczna** i ma uzasadnienie
ulokowane w konkretnym notebooku eksploracyjnym.

| Decyzja | Wartość / wybór | Uzasadnienie |
|---|---|---|
| Wejście modelu | wyłącznie kolumna `text` | `subject` 1:1 z klasą; `date` — niestandardowy format Fake; `title` — prefiksy clickbaitowe (EDA bloki 3, 5; leakage bloki 1, 5) |
| Etykieta | `1 = Real`, `0 = Fake` | konwencja projektowa (CLAUDE.md) |
| Cleaning | regex pipeline w `cleaning.py` | usuwa wszystkie markery z `02_leakage.ipynb` (sanity-check w `04_post_cleaning.ipynb`) |
| Filtr długości | `min_words = 10` | ~870 rekordów to scrapowe artefakty / puste teksty (EDA blok 4) |
| Deduplikacja | `drop_duplicates(text)` po cleaningu | ~6.4 tys. dokładnych duplikatów, dominuje Fake (EDA blok 2) |
| `max_seq_len` | **900 słów** | 95. percentyl po cleaningu (EDA blok 4 + post-cleaning blok 3) |
| Tokenizacja | whitespace + regex `[a-zA-Z]+`, lower-case | dopasowane do GloVe; bez subword/BPE (PyTorch-only stack) |
| Embedding | **GloVe 6B 300d**, frozen na początek | duża pokrywalność po cleaningu; wymóg projektu (BiLSTM+Attention) |
| OOV | jeden `<unk>` token, `<pad>` osobno | brak HF tokenizers — własny vocab z train splitu |
| Stack DL | **strict PyTorch** | bez TF/Keras/JAX, bez `Trainer`/Lightning (decyzja projektowa) |

## 6. Plan dalszych prac

Etap **EDA + cleaning** zamknięty. Dalsze kroki, w kolejności realizacji:

**E1. Split danych.** Stratyfikowany podział `news_cleaned.parquet` na train/val/test (70/15/15) z deterministycznym `random_state`. Zapis do `data/processed/splits/` (osobne parquet-y), żeby zagwarantować powtarzalność wyników między eksperymentami.

**E2. Vocabulary + matryca embeddingów.** Tokenizacja `[a-zA-Z]+` + lower-case (taka sama jak w EDA bloku 7), zbudowanie `Vocab` z train-split (`min_freq` do dostrojenia, ~2–5), wczytanie GloVe 6B 300d, zbudowanie `embedding_matrix` `[V × 300]`. Tokeny OOV → wektor `<unk>` (mean GloVe lub random-normal). Skrypt: `src/preprocessing/vocab.py`.

**E3. Dataset i DataLoader (PyTorch).** Klasa `NewsDataset(Dataset)` zwraca `(token_ids, length, label)`. Padding via `pad_sequence` w collate-fn. Batch size do dostrojenia (start: 32). Plik: `src/model/dataset.py`.

**E4. Model — BiLSTM + Attention.** 
- `nn.Embedding.from_pretrained(embedding_matrix, freeze=True)` (faza 1) → odmrożenie po stabilizacji.
- `nn.LSTM(input_size=300, hidden_size=128, bidirectional=True, batch_first=True)`.
- Attention head: dot-product / additive (Bahdanau) — do wyboru w eksperymentach.
- Klasyfikator: `Linear(2*hidden, 1)` + `BCEWithLogitsLoss`.
- Plik: `src/model/bilstm_attn.py`.

**E5. Pętla treningowa.** Własna pętla (bez Lightning / Trainer) z: AdamW, ReduceLROnPlateau, early stopping na val F1, gradient clipping, mixed precision (`torch.amp`), checkpointing co epokę. Pliki: `src/model/train.py`, `src/model/eval.py`.

**E6. Ewaluacja.** Test split: accuracy, F1, precision/recall, confusion matrix, ROC/PR. Notebook prezentacyjny: `notebooks/report/01_metrics_report.ipynb`.

**E7. Wyjaśnialność (XAI).** Wizualizacja wag atencji (heatmapa per token), ewentualnie integrated gradients. Notebook: `notebooks/report/02_xai_report.ipynb`.

**E8. Pisanie pracy.** Sekcje: dataset, EDA, leakage, cleaning, model, wyniki, XAI, dyskusja — wszystkie wnioski z tego raportu trafiają wprost do tekstu pracy.